In [1]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if not gpus:
    raise SystemError("GPU não encontrada! O treinamento foi abortado para evitar lentidão na CPU.")

try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ Infraestrutura validada. Treinamento travado na GPU: {gpus[0].name}")
except RuntimeError as e:
    print(f"Aviso de configuração de memória: {e}")

I0000 00:00:1780320911.479735    8579 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1780320914.014885    8579 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


SystemError: GPU não encontrada! O treinamento foi abortado para evitar lentidão na CPU.

In [2]:
import os, glob, sys

nvidia_libs = glob.glob(sys.prefix + "/lib/python3.11/site-packages/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs)
os.environ["KAGGLE_CACHE_DIR"] = os.path.join(os.getcwd(), "data")

import kagglehub
import tensorflow as tf

print("Baixando dataset do Kaggle...")
path = kagglehub.dataset_download("abdelghaniaaba/wildfire-prediction-dataset")
print(f"Dataset salvo em: {path}")
print("\nConteúdo da pasta principal:", os.listdir(path))

DATA_DIR = path
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print("\nCarregando Base de Treinamento:")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

print("\nCarregando Base de Validação:")
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "valid"),
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

Baixando dataset do Kaggle...


/home/joaovictor/projects/college/machine-learning/eco-orbit-deep-learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset salvo em: /home/joaovictor/.cache/kagglehub/datasets/abdelghaniaaba/wildfire-prediction-dataset/versions/1

Conteúdo da pasta principal: ['test', 'train', 'valid']

Carregando Base de Treinamento:
Found 30249 files belonging to 2 classes.


I0000 00:00:1780318272.055420   12005 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4147 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:08:00.0, compute capability: 7.5



Carregando Base de Validação:
Found 6300 files belonging to 2 classes.


In [ ]:
import os
import tensorflow as tf

print("Iniciando varredura por imagens corrompidas...")

arquivos_deletados = 0

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        file_path = os.path.join(root, file)

        try:
            image_content = tf.io.read_file(file_path)
            tf.io.decode_image(image_content, expand_animations=False)

        except Exception as e:
            print(f"Lixo encontrado e deletado: {file_path}")
            os.remove(file_path)
            arquivos_deletados += 1

print(f"\nLimpeza concluída! {arquivos_deletados} imagens corrompidas foram removidas.")

Iniciando varredura por imagens corrompidas...


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os


BATCH_SIZE = 16
IMG_SIZE = (224, 224)

In [ ]:
print("Montando Pipeline de Treino:")
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='binary'
)

print("\nMontando Pipeline de Validação:")
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'valid'),
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='binary'
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False


inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stopping = keras.callbacks.EarlyStopping(
    patience=3,
    restore_best_weights=True
)

print("Iniciando Treinamento...")
history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[early_stopping]
)

In [ ]:
model.save('modelo_wildfire.keras')
print("✅ Modelo treinado e salvo com sucesso como 'modelo_wildfire.keras'")